# DS 227 · Week 7 — Data exploration I
## Descriptive statistics and univariate EDA

Every code block from the lecture, in order, so you can run what you watched instead of
retyping it. The first slides use small tables typed into the cells; the **Case study**
cell loads the flood control data from the course API, so there is no file to download.

**How to use this.** Run the cells top to bottom the first time (Runtime → Run all). The
exercise cells list their tasks above the code: predict first, then run, then change one
thing and run again.

**Version differences.** The slides follow pandas 3. On pandas 2, text columns show as
`object` instead of `str`, and dates as `datetime64[ns]` instead of `datetime64[us]`.


In [ ]:
import numpy as np
import pandas as pd

---
## The lecture, cell by cell

Headings carry the slide numbers shown in the deck's counter.


### Slide 8 — Code on these slides runs in the browser


Expected output:

```
pandas version: 3.0.5
64-bit Python: True
city      str
pop     int64
dtype: object
```


In [ ]:
import io, sys

print("pandas version:", pd.__version__)
print("64-bit Python:", sys.maxsize > 2**32)

t = pd.read_csv(io.StringIO("city,pop\nCebu,964000"))
print(t.dtypes)

### Slide 10 — describe(): a full summary in one call


**Exercise.**

1. Predict the median (50%) before running: the middle of the 12 sorted scores.
2. Run the cell.
3. Change the last score, 95, to 195. Which lines of the summary change?


In [ ]:
scores = pd.Series([72, 85, 90, 60, 75, 88, 91, 55, 79, 83, 68, 95])

print(scores.describe().round(2))

### Slide 12 — Mean and median agree only for symmetric data


**Exercise.**

1. Predict both pairs of numbers.
2. Run the cell.
3. Change 120 to 1200. What happens to the mean, and to the median?


In [ ]:
tight  = pd.Series([10, 12, 11, 13, 12])
print(tight.mean(), tight.median())

skewed = pd.Series([10, 12, 11, 13, 120])
print(skewed.mean(), skewed.median())

### Slide 13 — Same mean, very different spreads


**Exercise.**

1. Predict which series has the larger standard deviation.
2. Run the cell.
3. Change wild to [30, 70, 50, 50, 50]. What happens to the SD and the range?


In [ ]:
calm = pd.Series([50, 51, 49, 50, 50])
print(calm.mean(), round(calm.std(), 1), calm.max() - calm.min())

wild = pd.Series([10, 90, 30, 70, 50])
print(wild.mean(), round(wild.std(), 1), wild.max() - wild.min())

### Slide 14 — A standard deviation, by hand


Expected output:

```
4000.0
1000.0
31.622776601683793
```


In [ ]:
wild = pd.Series([10, 90, 30, 70, 50])

print(((wild - wild.mean())**2).sum())
print(wild.var())       # 4000 / (5 - 1)
print(wild.std())       # square root of 1000

### Slide 15 — The flood data, loaded from the course API


Expected output:

```
(34079, 16)
```


In [ ]:
import io, requests

url  = "https://portal.latarak.com/api/flood/export.csv"
resp = requests.get(url, timeout=60)
df   = pd.read_csv(io.StringIO(resp.text))

# week 6's derived column, needed later today
for col in ["start_date", "completion_date"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")
df["duration"] = (df.completion_date - df.start_date).dt.days

print(df.shape)

### Slide 16 — describe() on 34,079 budgets


Expected output:

```
count          34,079
mean       46,952,340
std        48,594,319
min                 0
25%        14,649,406
50%        37,676,233
75%        67,549,270
max     1,447,499,996
Name: budget, dtype: float64
```


In [ ]:
pd.set_option("display.float_format", "{:,.0f}".format)
print(df.budget.describe())

### Slide 18 — The count line is the only sign of the omission


Expected output:

```
34079
27664.0
0.188
```


In [ ]:
print(len(df))
print(df.duration.describe()["count"])

# always compare the two
print(round(df.duration.isna().sum() / len(df), 3))

### Slide 21 — A standard deviation larger than the median


Expected output:

```
48594319
-50236298
```


In [ ]:
print(round(df.budget.std()))

# mean - 2*SD is NEGATIVE
print(46_952_340 - 2*48_594_319)

### Slide 22 — A skew of 5.13 indicates strong right skew


Expected output:

```
5.13
76.01
-0.0072
```


In [ ]:
print(round(df.budget.skew(), 2))
print(round(df.budget.kurtosis(), 2))

# for comparison, a normal sample:
bell = np.random.default_rng(0).normal(size=34079)
print(round(pd.Series(bell).skew(), 4))

### Slide 23 — Each percentile translates into a plain statement


Expected output:

```
0.05      1.9
0.25     14.6
0.50     37.7
0.75     67.5
0.90     96.5
0.99    212.1
Name: budget, dtype: float64
```


In [ ]:
pd.reset_option("display.float_format")   # undo the comma setting
p = df.budget.quantile([.05, .25, .5, .75, .9, .99])
print((p / 1e6).round(1))     # in millions of pesos

### Slide 28 — A histogram shows where values occur


**Exercise.**

1. Predict how many of the 12 incomes fall in the tallest bar.
2. Run the cell.
3. Change bins=6 to bins=3. What happens to the 60k income, and to the rest?


In [ ]:
import matplotlib.pyplot as plt

income = pd.Series([15, 18, 20, 22, 19, 21, 17, 60, 16, 23, 18, 20])

fig, ax = plt.subplots()
ax.hist(income, bins=6)
ax.set_xlabel("Income (k)")
ax.set_ylabel("Count")
plt.show()

### Slide 32 — Histogram of the budget column


In [ ]:
df.budget.hist(bins=50)
plt.show()     # 43% of projects in the first bar (under PHP 29M)

# readable version: clip, and say so in the caption
df.budget[df.budget < 2e8].hist(bins=50)
plt.show()

### Slide 37 — Skew 5.13 becomes −0.82


Expected output:

```
-0.82
7.47
29733984
```


In [ ]:
lb = np.log10(df.budget[df.budget > 0])
print(round(lb.skew(), 2))

# the log-mean, back in pesos
print(round(lb.mean(), 2))
print(round(10**lb.mean()))

### Slide 39 — Week 5’s IQR rule as a boxplot


Expected output:

```
146.9 798
```


In [ ]:
q1, q3 = df.budget.quantile([.25, .75])
fence = q3 + 1.5 * (q3 - q1)              # week 5: the upper fence
print(round(fence / 1e6, 1), (df.budget > fence).sum())

df.boxplot(column="budget", by="region")
plt.show()

### Slide 44 — Categories are counted, not averaged


**Exercise.**

1. Predict the share of Visayas.
2. Run the cell.
3. Replace the last line with print(r.describe()). What does describe() report for text?


In [ ]:
r = pd.Series(["Visayas"] * 6 + ["Mindanao"] * 4 + ["Luzon"] * 2,
              name="region")
print(r.value_counts())
print(r.value_counts(normalize=True).round(2))

### Slide 45 — On real data: 18 regions, 4,841 contractors


Expected output:

```
region
Region III                 0.159
National Capital Region    0.115
Region I                   0.103
Region IV-A                0.101
Name: proportion, dtype: float64
18 4841
Region III
```


In [ ]:
print(df.region.value_counts(normalize=True).round(3).head(4))
print(df.region.nunique(), df.contractor.nunique())

# mode = the most common value
print(df.region.mode()[0])

### Slide 50 — The fraction below a threshold, without binning


Expected output:

```
0.214
0.709
0.936
0.064 0.227
```


In [ ]:
print(round((df.budget < 1e7).mean(), 3))
print(round((df.budget < 5e7).mean(), 3))
print(round((df.budget < 1e8).mean(), 3))

# the projects of PHP 100M or more, and their share of the money
big = df.budget >= 1e8
print(round(big.mean(), 3), round(df.budget[big].sum() / df.budget.sum(), 3))

### Slide 51 — CV makes spread in pesos and days comparable


Expected output:

```
1.03
0.68
```


In [ ]:
def cv(x):
    return x.std() / x.mean()

print(round(cv(df.budget), 2))
print(round(cv(df.duration), 2))

### Slide 53 — Six lines that describe any numeric column


In [ ]:
col, threshold = df.budget, 1e8

col.isna().sum()           # what is missing
col.describe()             # centre + spread
col.skew()                 # which way it leans
col.quantile([.05, .95])   # the realistic range
(col < threshold).mean()   # ECDF at a point
col.hist(bins=30)          # look at it

---

*Data: DPWH flood control projects 2016–2026, mirrored by BetterGov.ph from the DPWH
transparency portal, CC0-1.0 (public domain). Served by the course API at* `/api/flood`.
